In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
data=pd.read_csv('TRAIN_FE.csv')
data.head()

,F01,F02,F03,F04,F05,F06,F07,F08,F09,F10,...,ratio_F32_F12,ratio_F33_F13,ratio_F34_F14,ratio_F35_F15,ratio_F36_F16,ratio_F37_F17,ratio_F38_F18,F19_residual,grpA_sum,grpB_sum
0,0.185570,0.004568,0.005362,0.003335,0.005415,0.004895,0.012764,0.120138,0.140450,3.361753,...,1.015246,0.960946,0.837119,0.818194,0.739435,0.801747,0.745842,-0.252623,0.482497,32.077725
1,0.369536,0.003983,0.003386,0.004902,0.007570,0.012136,0.118050,0.323925,0.132093,2.766117,...,10.000095,2.753256,0.923644,0.984414,0.856845,0.761899,0.817288,-0.758267,0.975581,31.117146
2,0.602510,0.008442,0.012961,0.012870,0.046885,0.115401,0.065688,0.306677,0.498805,4.521201,...,3.306491,2.600483,0.866374,0.869999,0.882478,0.774873,0.749903,0.187816,1.670239,37.275633
3,0.347957,0.064721,0.013611,0.011541,0.006492,0.008690,0.013192,0.164553,0.298665,3.170847,...,266.947549,117.136034,95.478884,39.237070,5.164514,0.781215,0.835150,0.073217,0.929422,25.055210
4,0.233653,0.012217,0.010088,0.022095,0.026040,0.015062,0.016063,0.084648,0.213367,8.150943,...,1.417105,0.926842,0.975356,0.812273,0.954541,0.811071,1.006131,0.832870,0.633233,43.676510


In [3]:
X=data.drop('Class',axis=1)
y=data['Class']

In [4]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,train_size=0.25,random_state=42)

In [19]:
from sklearn.ensemble import RandomForestClassifier
clf=RandomForestClassifier(n_estimators=100,random_state=42,n_jobs=-1,class_weight='balanced')
clf.fit(X_train,y_train)

RandomForestClassifier(class_weight='balanced', n_jobs=-1, random_state=42)

In [20]:
y_prob=clf.predict_proba(X_test)[:,1]
y_pred=clf.predict(X_test)

In [21]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report,roc_auc_score


In [26]:
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))
print(accuracy_score(y_test,y_pred))
auc=roc_auc_score(y_test,y_prob)
print('auc socre',auc)

              precision    recall  f1-score   support

           0       0.94      0.99      0.97     19800
           1       0.99      0.91      0.95     13032

    accuracy                           0.96     32832
   macro avg       0.97      0.95      0.96     32832
weighted avg       0.96      0.96      0.96     32832

[[19696   104]
 [ 1153 11879]]
0.9617141812865497
auc socre 0.9964559189190865


In [27]:
print("Train Accuracy:",clf.score(X_train,y_train))
print("Test accuracy:",clf.score(X_test,y_test))

Train Accuracy: 1.0
Test accuracy: 0.9617141812865497


In [28]:
## checking the features importance.
feature_importance=pd.Series(clf.feature_importances_)
feature_importance.sort_values(ascending=False).head(10)

8     0.059025
18    0.055612
36    0.048002
0     0.044612
47    0.044348
4     0.039058
7     0.035386
6     0.033996
5     0.032426
34    0.029579
dtype: float64

In [29]:
## Lets do Cross Validation of our model
from sklearn.model_selection import cross_val_score

score=cross_val_score(clf,X,y,cv=5,scoring='f1',n_jobs=-1)

In [30]:
print("CV scores:",score)
print("Mean CV AUC:",score.mean())

CV scores: [0.97661421 0.9740566  0.97853572 0.97252585 0.97648442]
Mean CV AUC: 0.9756433613592682


In [31]:
## checking for Leakage or overfitting.
y_suff=np.random.permutation(y)

scores=cross_val_score(
    clf,
    X,
    y_suff,
    cv=5,
    scoring='f1',
    n_jobs=-1
)


In [32]:
print(scores.mean())


0.14544128436033488


In [33]:
## Hyper parameter Tuning
from sklearn.model_selection import RandomizedSearchCV

param_grid={
    'n_estimators':[200,300,500],
    'max_depth':[10,15,20,None],
    'min_samples_split':[2,5,10],
    'min_samples_leaf':[1,3,5,10],
    'max_features':['sqrt','log2',0.5]
}

In [34]:
random_search=RandomizedSearchCV(
    clf,
    param_distributions=param_grid,
    n_iter=25,
    scoring='f1',
    cv=5,
    verbose=1,
    n_jobs=-1
)

In [35]:
random_search.fit(X_train,y_train)

Fitting 5 folds for each of 25 candidates, totalling 125 fits


RandomizedSearchCV(cv=5,
                   estimator=RandomForestClassifier(class_weight='balanced',
                                                    n_jobs=-1,
                                                    random_state=42),
                   n_iter=25, n_jobs=-1,
                   param_distributions={'max_depth': [10, 15, 20, None],
                                        'max_features': ['sqrt', 'log2', 0.5],
                                        'min_samples_leaf': [1, 3, 5, 10],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [200, 300, 500]},
                   scoring='f1', verbose=1)

In [36]:
print("Best Params:",random_search.best_params_)
print("Best CV AUC:",random_search.best_score_)

Best Params: {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 0.5, 'max_depth': 20}
Best CV AUC: 0.9437972194285763


In [38]:
y_prob_hpt=clf.predict_proba(X_test)[:,1]
y_pred_hpt=random_search.predict(X_test)

In [39]:
print(classification_report(y_test,y_pred_hpt))
print(confusion_matrix(y_test,y_pred_hpt))
print(accuracy_score(y_test,y_pred_hpt))
auc=roc_auc_score(y_test,y_prob_hpt)
print('auc socre',auc)

              precision    recall  f1-score   support

           0       0.95      0.99      0.97     19800
           1       0.98      0.92      0.95     13032

    accuracy                           0.96     32832
   macro avg       0.97      0.96      0.96     32832
weighted avg       0.96      0.96      0.96     32832

[[19607   193]
 [ 1008 12024]]
0.9634198343079922
auc socre 0.9964559189190865


## Let's Go with Xgboost

In [40]:
from xgboost import XGBClassifier

In [44]:
xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='aucpr',
    random_state=42
)

xgb.fit(X_train, y_train)

y_prob = xgb.predict_proba(X_test)[:,1]

print("AUC:", roc_auc_score(y_test, y_prob))

AUC: 0.9969182153021933


In [45]:
y_pred=xgb.predict(X_test)

In [46]:
print(accuracy_score(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

0.9757858187134503
[[19587   213]
 [  582 12450]]
              precision    recall  f1-score   support

           0       0.97      0.99      0.98     19800
           1       0.98      0.96      0.97     13032

    accuracy                           0.98     32832
   macro avg       0.98      0.97      0.97     32832
weighted avg       0.98      0.98      0.98     32832



In [47]:
# lets try hyper parameter tuning
param_dist={
    "max_depth":[4,6,8],
    "learning_rate":[0.01,0.05,0.1],
    "n_estimators":[300,500,800],
    "subsample":[0.7,0.8,0.9],
    "colsample_bytree":[0.6,0.8,1]
}

In [50]:
from sklearn.model_selection import RandomizedSearchCV
random_search=RandomizedSearchCV(
    xgb,
    param_dist,
    n_iter=25,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    verbose=1
    )

In [51]:
random_search.fit(X_train,y_train)

Fitting 5 folds for each of 25 candidates, totalling 125 fits


RandomizedSearchCV(cv=5,
                   estimator=XGBClassifier(base_score=None, booster=None,
                                           callbacks=None,
                                           colsample_bylevel=None,
                                           colsample_bynode=None,
                                           colsample_bytree=0.8, device=None,
                                           early_stopping_rounds=None,
                                           enable_categorical=False,
                                           eval_metric='aucpr',
                                           feature_types=None,
                                           feature_weights=None, gamma=None,
                                           grow_policy=None,
                                           importance_type=None,
                                           interaction_constra...
                                           max_leaves=None,
                                           min_child_weight=None, missing=nan,
                                           monotone_constraints=None,
                                           multi_strategy=None,
                                           n_estimators=300, n_jobs=None,
                                           num_parallel_tree=None, ...),
                   n_iter=25, n_jobs=-1,
                   param_distributions={'colsample_bytree': [0.6, 0.8, 1],
                                        'learning_rate': [0.01, 0.05, 0.1],
                                        'max_depth': [4, 6, 8],
                                        'n_estimators': [300, 500, 800],
                                        'subsample': [0.7, 0.8, 0.9]},
                   scoring='f1', verbose=1)

In [52]:
y_pred_cv=random_search.predict(X_test)
y_prob_cv=random_search.predict_proba(X_test)[:,1]

In [53]:
print(accuracy_score(y_test,y_pred_cv))
print(confusion_matrix(y_test,y_pred_cv))
print(classification_report(y_test,y_pred_cv))
print("AUC:", roc_auc_score(y_test, y_prob_cv))

0.9789839181286549
[[19631   169]
 [  521 12511]]
              precision    recall  f1-score   support

           0       0.97      0.99      0.98     19800
           1       0.99      0.96      0.97     13032

    accuracy                           0.98     32832
   macro avg       0.98      0.98      0.98     32832
weighted avg       0.98      0.98      0.98     32832

AUC: 0.99788041944925


In [54]:
#Cross Validating our model
from sklearn.model_selection import cross_val_score

score=cross_val_score(xgb,X_train,y_train,cv=5,scoring='f1')

print(score)
print(score.mean())

[0.96666667 0.97095436 0.95725467 0.96114764 0.97217288]
0.9656392423423842


## Lets's Go with LGBMClassifier

In [5]:
from lightgbm import LGBMClassifier

In [6]:
lgb_model=LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=-1,
    class_weight='balanced',
    random_state=42
)

In [7]:
lgb_model.fit(X_train,y_train)

[LightGBM] [Info] Number of positive: 4279, number of negative: 6665
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003301 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12495
[LightGBM] [Info] Number of data points in the train set: 10944, number of used features: 49
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


LGBMClassifier(class_weight='balanced', learning_rate=0.05, n_estimators=500,
               random_state=42)

In [8]:
y_prob=lgb_model.predict_proba(X_test)[:,1]
y_pred=lgb_model.predict(X_test)

In [12]:
from sklearn.metrics import accuracy_score,classification_report,roc_auc_score,confusion_matrix,f1_score


In [10]:
print(accuracy_score(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))
print("AUC:", roc_auc_score(y_test, y_prob))

0.9781310916179338
[[19566   234]
 [  484 12548]]
              precision    recall  f1-score   support

           0       0.98      0.99      0.98     19800
           1       0.98      0.96      0.97     13032

    accuracy                           0.98     32832
   macro avg       0.98      0.98      0.98     32832
weighted avg       0.98      0.98      0.98     32832

AUC: 0.997320286970379


In [13]:
param_grid = {
    "num_leaves":[31,63,127],
    "learning_rate":[0.01,0.03,0.05,0.07],
    "n_estimators":[400,600,800],
    "subsample":[0.7,0.8,0.9],
    "colsample_bytree":[0.7,0.8,0.9]
}

In [14]:
from sklearn.model_selection import RandomizedSearchCV

random_search=RandomizedSearchCV(
   lgb_model,
   param_grid,
   n_iter=25,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    verbose=1 
)

In [15]:
random_search.fit(X_train,y_train)

Fitting 5 folds for each of 25 candidates, totalling 125 fits
[LightGBM] [Info] Number of positive: 4279, number of negative: 6665
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003729 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12495
[LightGBM] [Info] Number of data points in the train set: 10944, number of used features: 49
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -i

RandomizedSearchCV(cv=5,
                   estimator=LGBMClassifier(class_weight='balanced',
                                            learning_rate=0.05,
                                            n_estimators=500, random_state=42),
                   n_iter=25, n_jobs=-1,
                   param_distributions={'colsample_bytree': [0.7, 0.8, 0.9],
                                        'learning_rate': [0.01, 0.03, 0.05,
                                                          0.07],
                                        'n_estimators': [400, 600, 800],
                                        'num_leaves': [31, 63, 127],
                                        'subsample': [0.7, 0.8, 0.9]},
                   scoring='f1', verbose=1)

In [16]:
y_prob_hp=random_search.predict_proba(X_test)[:,1]
y_pred_hp=random_search.predict(X_test)

In [17]:
print(accuracy_score(y_test,y_pred_hp))
print(confusion_matrix(y_test,y_pred_hp))
print(classification_report(y_test,y_pred_hp))
print("AUC:", roc_auc_score(y_test, y_prob_hp))

0.9824561403508771
[[19647   153]
 [  423 12609]]
              precision    recall  f1-score   support

           0       0.98      0.99      0.99     19800
           1       0.99      0.97      0.98     13032

    accuracy                           0.98     32832
   macro avg       0.98      0.98      0.98     32832
weighted avg       0.98      0.98      0.98     32832

AUC: 0.9985347683402471


In [18]:
from sklearn.model_selection import cross_val_score

score=cross_val_score(lgb_model,X_train,y_train,cv=5,scoring='f1')

print(score)
print(score.mean())

[LightGBM] [Info] Number of positive: 3423, number of negative: 5332
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004786 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12495
[LightGBM] [Info] Number of data points in the train set: 8755, number of used features: 49
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Info] Number of positive: 3423, number of negative: 5332
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003277 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12495
[LightGBM] [Info] Number of data points in the train set: 8755, number of used features: 49
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Info] Nu

In [55]:
Test_data=pd.read_csv('TEST_FE.csv')

In [56]:
Test_data.head()

,F01,F02,F03,F04,F05,F06,F07,F08,F09,F10,...,ratio_F32_F12,ratio_F33_F13,ratio_F34_F14,ratio_F35_F15,ratio_F36_F16,ratio_F37_F17,ratio_F38_F18,F19_residual,grpA_sum,grpB_sum
0,0.277497,0.011515,0.009359,0.011706,0.015613,0.063679,0.021114,0.152000,0.220904,3.728738,...,0.950356,0.854783,0.893513,0.840863,0.772959,0.853771,0.892606,-0.372332,0.783387,30.336097
1,0.158754,0.013861,0.020935,0.002451,0.005939,0.006830,0.006934,0.117209,0.103432,2.937431,...,1.014439,1.753128,0.883034,0.766611,0.782498,0.741232,0.859110,-0.308266,0.436345,27.529045
2,0.504418,0.009106,0.007395,0.009073,0.029486,0.086782,0.122499,0.179376,0.445695,4.392277,...,1.752702,1.081435,0.858805,0.858350,0.877732,0.782220,0.866677,-0.204331,1.393830,33.956340
3,0.556345,0.005370,0.005670,0.005167,0.013731,0.014570,0.014776,0.522296,0.189882,2.684893,...,52.292983,44.124158,0.835944,0.840750,0.993413,0.919108,0.834301,-0.944789,1.327807,31.429487
4,0.128356,0.010014,0.016144,0.003891,0.004472,0.005297,0.007538,0.075874,0.101176,3.240038,...,0.974324,0.879949,0.825545,0.769209,0.733115,0.771655,0.884465,-0.217636,0.352762,29.876096


In [74]:
## Final Training Data.
Test_prediction=random_search.predict(Test_data)

In [75]:
test_df=pd.DataFrame(Test_prediction)

In [ ]:
df_test=pd.read_csv('TEST.csv')
df_test.head()
test_df['ID']=df_test['ID']

In [ ]:
test_df['Class']=test_df[0]

In [80]:
test_df.head()

,0,ID,Class
0,1,1,1
1,0,2,0
2,1,3,1
3,0,4,0
4,0,5,0


In [81]:
test_df.drop(0,axis=1,inplace=True)

In [82]:
test_df.head()

,ID,Class
0,1,1
1,2,0
2,3,1
3,4,0
4,5,0


,ID,F01,F02,F03,F04,F05,F06,F07,F08,F09,...,F38,F39,F40,F41,F42,F43,F44,F45,F46,F47
0,1,0.277497,0.011515,0.009359,0.011706,0.015613,0.063679,0.021114,0.152000,0.220904,...,2.974426,0.009399,0.015668,0.001999,-0.000937,0.010605,-0.001117,0.043216,-0.000077,-0.040124
1,2,0.158754,0.013861,0.020935,0.002451,0.005939,0.006830,0.006934,0.117209,0.103432,...,2.619811,0.029324,-0.003211,0.037691,-0.083697,0.006595,0.044507,0.007523,-0.000283,-0.007837
2,3,0.504418,0.009106,0.007395,0.009073,0.029486,0.086782,0.122499,0.179376,0.445695,...,4.411972,-0.171520,-0.866895,-0.070657,0.034070,-0.000493,-0.011417,-0.004844,-0.034235,-0.001832
3,4,0.556345,0.005370,0.005670,0.005167,0.013731,0.014570,0.014776,0.522296,0.189882,...,2.709162,0.044236,19.880789,-2.144453,-1.792552,-0.034137,-0.051971,-0.085069,-0.000110,0.045889
4,5,0.128356,0.010014,0.016144,0.003891,0.004472,0.005297,0.007538,0.075874,0.101176,...,2.872020,-0.056908,0.006535,0.070059,-0.007989,0.004089,-0.000206,-0.011331,0.000441,-0.025263


In [78]:
test_df.head()

,0,ID
0,1,1
1,0,2
2,1,3
3,0,4
4,0,5


In [83]:
test_df.to_csv('FINAL.csv',index=False)